In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import duckdb

In [3]:
con = duckdb.connect(r'C:\Users\marzieh\Documents\GitHub\Tennis-project\data\tennis.duckdb', read_only=True)
con.execute("SHOW TABLES").df()

,name
0,_build_info
1,_import_audit
2,_schema_audit
3,_source_files
4,game_point_by_point
5,match_away_score
6,match_away_team
7,match_event
8,match_home_score
9,match_home_team


In [4]:
query = """
    SELECT p.match_id,
           set_num,
           MAX(game_num) AS game_num,
           h.gender AS h_gender,
           a.gender AS gender
    FROM match_home_team as h
    INNER JOIN match_away_team as a
    ON h.match_id = a.match_id
    INNER JOIN power as p
    ON p.match_id = h.match_id
    GROUP BY p.match_id, set_num, h.gender, a.gender
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY p.match_id
        ORDER BY p.match_id
    ) = 1
    
"""
df = con.execute(query).df()
df

,match_id,set_num,game_num,h_gender,gender
0,12127637,2,10,M,M
1,12127782,2,9,F,F
2,12109843,2,8,M,M
3,12128163,2,7,M,M
4,12131400,2,9,F,F
...,...,...,...,...,...
8796,12205253,1,6,F,F
8797,12002073,1,8,F,F
8798,12104192,2,8,M,M
8799,12199628,1,6,M,M


In [5]:
t1 = df['gender'] == df['h_gender']
t1.value_counts()

True     8779
False      22
Name: count, dtype: int64

In [6]:
df_clean = df[df['gender']==df['h_gender']].drop(columns=['h_gender'])
df_clean

,match_id,set_num,game_num,gender
0,12127637,2,10,M
1,12127782,2,9,F
2,12109843,2,8,M
3,12128163,2,7,M
4,12131400,2,9,F
...,...,...,...,...
8796,12205253,1,6,F
8797,12002073,1,8,F
8798,12104192,2,8,M
8799,12199628,1,6,M


In [7]:
df_clean.groupby("gender")['game_num'].mean().rename("average_game_num")

gender
F    8.975280
M    9.404173
Name: average_game_num, dtype: float64